# Day 4.2 — Single-Reviewer Baseline
The simplest system that could work: one reviewer, one call, no tools, no supervisor. Everything
later has to beat it. Live model and scripted stand-in share one request shape and one validator.

### Step 1 — One request shape, one schema

`review_messages` builds the only prompt in this notebook; `Role:` is what narrows a reviewer
later. The reply must be JSON in a fixed shape, so we ask with a strict schema: a prompt raises
the probability, the schema and the validator make it safe.

In [ ]:
class ModelFinding(BaseModel):
    """What the model is allowed to send us: the claim, without ids or provenance."""
    category: Literal["correctness", "security", "maintainability"]
    line: int = Field(ge=1)
    title: str
    evidence: str
    severity: Literal["low", "medium", "high", "critical"]
    recommendation: str

class FindingsReply(BaseModel):
    findings: list[ModelFinding]

FINDINGS_FORMAT = {"type": "json_schema", "json_schema": {
    "name": "FindingsReply", "strict": True,
    "schema": make_strict(FindingsReply.model_json_schema())}}

def review_messages(source, role):
    """The ONE request shape every reviewer sees - live model or scripted stand-in."""
    scope = ("all correctness, security and maintainability concerns" if role == "general"
             else f"only {role} concerns")
    return [
        {"role": "system", "content":
            f"You are a bounded engineering code reviewer. Role: {role}. Report {scope}. "
            "Every finding needs the 1-based line number and the exact source line as evidence. "
            "Report at most 10 findings. Do not invent code."},
        {"role": "user", "content": "Review this Python file:\n\n" + source},
    ]

print(review_messages(SOURCE, "security")[0]["content"])

### Step 2 — Validate before you believe

`parse_findings` turns a reply into `Finding` records or raises. It also enforces the role
contract: a security specialist returning a maintainability finding has broken its bounds, and an
error beats silently widening the role.

In [ ]:
def parse_findings(content, role, max_findings=10):
    """Validate the model's JSON and stamp ids and provenance onto each record."""
    text = content.strip()
    if text.startswith("```"):                                    # models like to fence JSON
        text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text, flags=re.I)
    reply = FindingsReply.model_validate_json(text)               # shape, types, ranges
    if len(reply.findings) > max_findings:
        raise ValueError(f"reviewer returned {len(reply.findings)} findings, cap is {max_findings}")
    findings = []
    for index, item in enumerate(reply.findings, 1):
        if role != "general" and item.category != role:           # the role contract
            raise ValueError(f"{role} reviewer returned a {item.category} finding")
        findings.append(Finding(id=f"MODEL-{role[:3].upper()}-{item.line}-{index}",
                                reviewer=f"{role}_reviewer", **item.model_dump()))
    return findings

# A reply that breaks the contract is rejected loudly, not trusted quietly.
for label, bad in [("wrong category for the role",
                    '{"findings":[{"category":"correctness","line":7,"title":"t","evidence":"e","severity":"low","recommendation":"r"}]}'),
                   ("line number is not a number",
                    '{"findings":[{"category":"security","line":"three","title":"t","evidence":"e","severity":"low","recommendation":"r"}]}')]:
    try:
        parse_findings(bad, "security")
    except (ValidationError, ValueError) as error:
        print(f"{label:<30} -> rejected: {str(error).splitlines()[0][:70]}")

### Step 3 — One reviewer call, whichever model is behind it

`call_reviewer` is the whole reviewer: build the request, send it through a model, validate the
reply. `model` is any function with `chat`'s signature — real, mock, or deliberately broken.

In [ ]:
def call_reviewer(source, role, model=chat):
    """Return (findings, usage). Usage is what the model reported, never an estimate of ours."""
    reply = model(review_messages(source, role), response_format=FINDINGS_FORMAT)
    return parse_findings(reply["content"], role), reply["usage"]

print("call_reviewer ready. It works with any model function:", chat.__name__, "or a stand-in.")

### Step 4 — The scripted reviewer, and why its blind spots are a parameter

In mock mode the reviewer is **scripted, not intelligent**: its blind spots are chosen so the
result is reproducible anywhere. Two worlds — `blind_spots`, where the generalist misses a lot,
and `strong_generalist`, where it nearly matches the team. That is how we compare *architectures*
while holding the reviewer constant.

In [ ]:
import time

# Every seeded defect: id -> (category, source needle, title, severity, recommendation).
# The needle is how a scripted reviewer "locates" a defect in the artifact.
RULES = {
    "DEF-COR-01": ("correctness", "quantity", "Negative quantities are accepted", "high",
                   "Validate that quantity is a positive integer."),
    "DEF-COR-02": ("correctness", "subtotal - 20", "Flat discount can make the total negative", "medium",
                   "Use a percentage and reject invalid totals."),
    "DEF-COR-03": ("correctness", "/ len(items)", "Empty input causes division by zero", "medium",
                   "Define empty-list behaviour before dividing."),
    "DEF-SEC-01": ("security", "ADMIN_TOKEN", "Credential-like token is hardcoded", "high",
                   "Load secrets from an injected secret store."),
    "DEF-SEC-02": ("security", "SELECT * FROM", "SQL query uses string concatenation", "critical",
                   "Use a parameterised query."),
    "DEF-SEC-03": ("security", "eval(", "Untrusted expression may reach eval", "critical",
                   "Replace eval with an allow-listed parser."),
    "DEF-SEC-04": ("security", 'user.get("token")', "Sensitive token is written to logs", "high",
                   "Never log tokens; redact sensitive fields."),
    "DEF-MNT-01": ("maintainability", "audit=[]", "Mutable default argument retains state", "medium",
                   "Default to None and create the list inside."),
    "DEF-MNT-02": ("maintainability", "except Exception", "Broad exception hides failures", "medium",
                   "Catch only the exceptions you expect."),
}
ALL_IDS = sorted(RULES)

SCENARIOS = {
    # World A: one generalist has real blind spots; narrow roles cover them.
    "blind_spots": {
        "general":         ["DEF-COR-03", "DEF-SEC-01", "DEF-SEC-02", "DEF-SEC-03", "DEF-MNT-01"],
        "correctness":     ["DEF-COR-01", "DEF-COR-02", "DEF-COR-03"],
        "security":        ["DEF-SEC-01", "DEF-SEC-02", "DEF-SEC-03", "DEF-SEC-04"],
        "maintainability": ["DEF-MNT-01", "DEF-MNT-02"],
    },
    # World B: the generalist is strong. It misses only the subtle business rule (DEF-COR-02)
    # - and so does the correctness specialist, because narrowing a prompt does not create
    # knowledge the reviewer never had.
    "strong_generalist": {
        "general":         [i for i in ALL_IDS if i != "DEF-COR-02"],
        "correctness":     ["DEF-COR-01", "DEF-COR-03"],
        "security":        ["DEF-SEC-01", "DEF-SEC-02", "DEF-SEC-03", "DEF-SEC-04"],
        "maintainability": ["DEF-MNT-01", "DEF-MNT-02"],
    },
}

def scripted_findings(source, role, scenario=MOCK_SCENARIO):
    """Replaces the stub at the top: the mock model's reviewer, in schema shape."""
    text_lines = source.splitlines()
    out = []
    for defect_id in SCENARIOS[scenario][role]:
        category, needle, title, severity, fix = RULES[defect_id]
        if needle not in source:
            continue
        line = next(i for i, text in enumerate(text_lines, 1) if needle in text)
        out.append({"category": category, "line": line, "title": title,
                    "evidence": text_lines[line - 1].strip(), "severity": severity,
                    "recommendation": fix})            # note: no DEF- id leaks to the reviewer
    return sorted(out, key=lambda item: item["line"])

def make_mock_model(scenario, latency_s=0.0):
    """A stand-in for chat(): same signature, scripted findings, honest usage numbers."""
    def scripted(messages, tools=None, response_format=None):
        role, source = review_request(messages)
        if latency_s:
            time.sleep(latency_s)                      # stands in for a network round trip
        payload = {"findings": scripted_findings(source, role, scenario)}
        return _mock_reply(json.dumps(payload), sum(len(m["content"]) for m in messages))
    return scripted

for name, roles in SCENARIOS.items():
    print(f"{name:<18} general reviewer can see {len(roles['general'])} of 9 defects")

### Step 5 — Run the baseline and read its telemetry

`run_single_reviewer` makes exactly one call and records what it cost. The tokens come from the
model's own usage report: a table of estimates looks identical to a table of measurements.

In [ ]:
from dataclasses import dataclass, field
from time import perf_counter

@dataclass
class ReviewRun:
    """One end-to-end run of one review SYSTEM, with its measured telemetry."""
    system: str
    findings: list = field(default_factory=list)
    model_calls: int = 0
    prompt_tokens: int = 0
    completion_tokens: int = 0
    elapsed_ms: float = 0.0
    raw_findings: int = 0            # findings produced before any merging
    merged_duplicates: int = 0       # how many the supervisor collapsed
    dropped_over_cap: int = 0        # how many the supervisor truncated away
    trace: list = field(default_factory=list)

    @property
    def total_tokens(self):
        return self.prompt_tokens + self.completion_tokens

def run_single_reviewer(source, model):
    """System 1: ask one reviewer about everything, and report exactly what it said."""
    start = perf_counter()
    findings, usage = call_reviewer(source, "general", model)
    return ReviewRun(system="single_reviewer", findings=findings, model_calls=1,
                     prompt_tokens=usage["prompt_tokens"], completion_tokens=usage["completion_tokens"],
                     elapsed_ms=(perf_counter() - start) * 1000, raw_findings=len(findings),
                     trace=[{"step": "general_reviewer", "count": len(findings), "usage": usage}])

reviewer_blind = make_mock_model("blind_spots")          # the scripted reviewer of world A
single = run_single_reviewer(SOURCE, reviewer_blind)

print("System      :", single.system, "| model calls:", single.model_calls)
print("Tokens      :", single.total_tokens, "(prompt", single.prompt_tokens,
      "+ completion", single.completion_tokens, ") as reported by the model")
print("Elapsed ms  : %.2f\n" % single.elapsed_ms)
for finding in single.findings:
    print(f"line {finding.line:>3} | {finding.category:<15} | {finding.severity:<8} | {finding.title}")

### Step 6 — Score it, and see the shape of the hole

`missed` is the list this day exists to shrink — or to prove we cannot shrink it economically.

In [ ]:
baseline_row = score(single.findings)
print("Found          : %d / %d" % (baseline_row["found"], baseline_row["known_defects"]))
print("Recall         :", baseline_row["recall"])
print("False positives:", baseline_row["false_positives"], "| duplicates:", baseline_row["duplicates"])
print("Missed         :", baseline_row["missed"])
print("\nThree of the four misses are security or correctness problems this reviewer never looks at.")

### Step 7 — The live path, and what happens when it fails

The same `call_reviewer`, now through `chat`: real with a key, mock without one. A provider error
must never stop a class, so the reviewer falls back and says so in one line.

In [ ]:
def review_with_fallback(source, role, model=chat, scenario="blind_spots"):
    """Try the given model; on ANY failure print one line and use the scripted reviewer."""
    try:
        return call_reviewer(source, role, model)
    except Exception as error:                    # deliberate classroom safety net
        print(f"  reviewer failed for role={role} ({type(error).__name__}: {error}); "
              "falling back to the scripted reviewer")
        return call_reviewer(source, role, make_mock_model(scenario))

findings, usage = review_with_fallback(SOURCE, "general")
print("Mode           :", "LIVE model" if LIVE else "mock model")
print("Findings       :", len(findings), "| usage reported:", usage)
print("Valid records  :", all(isinstance(f, Finding) for f in findings), "- same contract either way")

# Prove the safety net works, using a model that always fails.
def broken_model(messages, tools=None, response_format=None):
    raise RuntimeError("503 provider unavailable")

recovered, _ = review_with_fallback(SOURCE, "general", model=broken_model)
print("After the failure:", len(recovered), "findings - the lesson continues")

### Try it yourself

Swap in the `strong_generalist` reviewer: same architecture, one call, no specialists. How many
of the 9 defects will it find? Write your number down, then run the worked solution.

In [ ]:
# --- Worked solution ---------------------------------------------------------------
# Same architecture (run_single_reviewer), different reviewer. Only the blind spots move.
reviewer_strong = make_mock_model("strong_generalist")
strong_row = score(run_single_reviewer(SOURCE, reviewer_strong).findings)

print("blind_spots       reviewer found %d/9  (missed %s)" % (baseline_row["found"], baseline_row["missed"]))
print("strong_generalist reviewer found %d/9  (missed %s)" % (strong_row["found"], strong_row["missed"]))
print("\nBoth runs made 1 model call. The architecture did not change; the reviewer did.")
print("Remember this the moment someone proposes adding agents.")

### Checkpoint

**1. Why does every reviewer — mock, live, and the broken one — go through the same `call_reviewer` contract?**

<details><summary>Show answer</summary>

So we can change one variable at a time: a difference in results then comes from the architecture or the reviewer, never from a rewritten pipeline.

</details>

**2. The run reports `prompt_tokens` from the model instead of estimating them from the file's length. Why does that matter?**

<details><summary>Show answer</summary>

An estimate in a results table looks exactly like a measurement. If a step that made no call shows a plausible token count, every cost comparison on that table is fiction.

</details>

### Recap

- **Limitation seen:** one general reviewer found 5 of 9 defects and missed a whole category's worth.
- **Layer added:** one request shape, one validator, one reviewer contract, with a fallback.
- **Evidence:** 5/9 recall from 1 call, and a forced provider failure that did not stop the run.